# RQ2 — Uniform + Resource-Preserving Geo extension

Reuse the clean protocol-v2 Resource and Pure-SW branches. GPU 0 trains Uniform. GPU 1 builds the frozen-state exact-face gate and trains RP-Geo only if the conservative gate returns `GO`. Validation/test accuracy never enters the gate.

## Required inputs

- CIFAR-100 with `cifar-100-python/{train,test,meta}`.
- Gate-A artifact containing `gate_a_summary.json`.
- Completed protocol-v2 output containing common epoch 10 plus Resource and Pure-SW epoch 100.
- Kaggle secret `github_token`.

In [ ]:
import os, subprocess, sys, json, time, zipfile, importlib, hashlib
from pathlib import Path
from IPython.display import display
from kaggle_secrets import UserSecretsClient
github_token = UserSecretsClient().get_secret('github_token')
assert github_token, 'Missing Kaggle secret github_token'
PROJECT_ROOT = Path('/kaggle/working/new-pruning')
askpass = Path('/kaggle/working/.github_git_askpass.py')
askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
askpass.chmod(0o700)
env = os.environ.copy(); env.update({'GIT_ASKPASS':str(askpass),'GIT_TERMINAL_PROMPT':'0','GITHUB_TOKEN_RUNTIME':github_token})
try:
    command = ['git','-C',str(PROJECT_ROOT),'pull','--ff-only'] if (PROJECT_ROOT/'.git').is_dir() else ['git','clone','https://github.com/duyh80456-code/new-pruning.git',str(PROJECT_ROOT)]
    subprocess.run(command, env=env, check=True)
finally:
    askpass.unlink(missing_ok=True); github_token = None
os.chdir(PROJECT_ROOT); sys.path.insert(0, str(PROJECT_ROOT))
import torch
assert torch.cuda.device_count() == 2, f'Select T4 x2; detected {torch.cuda.device_count()}'
GPU_IDS = (0,1)
GIT_COMMIT = subprocess.run(['git','rev-parse','HEAD'],capture_output=True,text=True,check=True).stdout.strip()
print('Commit:', GIT_COMMIT)

## Resolve and audit the frozen protocol-v2 inputs

In [ ]:
import rq2_e2e_pairwise_pilot as pilot
import scripts.run_e2e_pairwise_pilot as runner
pilot = importlib.reload(pilot); runner = importlib.reload(runner)
INPUT_ROOT = Path('/kaggle/input')
DATASET_ROOT = pilot.find_cifar100_root(INPUT_ROOT)
GATE_A_SUMMARY = pilot.find_gate_a_summary(INPUT_ROOT)
ROOT = pilot.materialize_progress(
    INPUT_ROOT, '/kaggle/working/e2e_pairwise_pilot_v2',
    '/kaggle/working/materialized-rpgeo-protocol-v2'
)
required_prior = [
 ROOT/'frozen_protocol.json', ROOT/'resolved_config.yaml', ROOT/'common_warmup/epoch_010.pt',
 ROOT/'resource/checkpoints/epoch_100.pt', ROOT/'pure_sw/checkpoints/epoch_100.pt',
 ROOT/'resource/training_provenance.json', ROOT/'pure_sw/training_provenance.json',
]
missing = [str(path) for path in required_prior if not path.is_file()]
assert not missing, f'Incomplete protocol-v2 input: {missing}'
base_protocol = json.loads((ROOT/'frozen_protocol.json').read_text())
assert base_protocol.get('protocol_version') == 2
for method in ('resource','pure_sw'):
    provenance = json.loads((ROOT/method/'training_provenance.json').read_text())
    assert provenance['policy_geometry_source'] == 'training_split_disjoint_from_bn_calibration'
    assert provenance['validation_used_to_build_policy'] is False
    assert provenance['test_used_to_build_policy'] is False
print('CIFAR-100:', DATASET_ROOT)
print('Gate A:', GATE_A_SUMMARY)
print('Protocol-v2 root:', ROOT)

## Freeze the exact-face RP-Geo extension

In [ ]:
extension_protocol = {
 'status':'FROZEN_BEFORE_RPGEO_EXTENSION', 'protocol_version':1, 'seed':3,
 'reused_methods':['resource','pure_sw'], 'new_methods':['uniform','resource_geo'],
 'resource_retention':1.0, 'exact_face_constraint':'equality_no_epsilon_slack',
 'rpgeo_objective':'maximize_squared_sliced_W1_on_resource_optimal_face',
 'refresh_states':[10,20,30,40,50,60,70,80,90],
 'fixed_marginal':1/7, 'validation_used_to_build_policy':False,
 'test_used_to_build_policy':False, 'accuracy_used_by_gate':False,
 'base_protocol_sha256':hashlib.sha256((ROOT/'frozen_protocol.json').read_bytes()).hexdigest(),
 'common_epoch10_sha256':pilot._sha256(ROOT/'common_warmup/epoch_010.pt'),
 'resource_epoch100_sha256':pilot._sha256(ROOT/'resource/checkpoints/epoch_100.pt'),
 'pure_sw_epoch100_sha256':pilot._sha256(ROOT/'pure_sw/checkpoints/epoch_100.pt'),
 'gate_a_sha256':hashlib.sha256(GATE_A_SUMMARY.read_bytes()).hexdigest(),
 'git_commit':GIT_COMMIT,
}
protocol_path = ROOT/'rpgeo_extension_protocol.json'
if protocol_path.exists():
    previous = json.loads(protocol_path.read_text())
    keys = [key for key in extension_protocol if key != 'git_commit']
    assert all(previous.get(key) == extension_protocol[key] for key in keys)
else:
    protocol_path.write_text(json.dumps(extension_protocol,indent=2)+'\n')
print(json.dumps(extension_protocol,indent=2))

## Run the two-lane extension

GPU 0 trains Uniform. GPU 1 extracts missing frozen diagnostics, runs the CPU exact-face gate, and trains RP-Geo only on `GO`. Mixed evidence returns `REVIEW_REQUIRED` and does not silently authorize training.

In [ ]:
started = time.perf_counter()
result, uniform_runtime, diagnostic_runtime = runner.run_rpgeo_extension(
    ROOT, DATASET_ROOT, GATE_A_SUMMARY, gpu_ids=GPU_IDS
)
print(json.dumps(result,indent=2))
display(uniform_runtime)
if not diagnostic_runtime.empty: display(diagnostic_runtime)
print(f'Elapsed: {(time.perf_counter()-started)/3600:.2f} h')

## Finalize when RP-Geo passed the frozen gate

In [ ]:
import pandas as pd
gate = json.loads((ROOT/'rpgeo_offline_gate/rpgeo_offline_gate.json').read_text())
display(pd.read_csv(ROOT/'rpgeo_offline_gate/rpgeo_offline_gate.csv'))
if gate['decision'] == 'GO':
    decision = pilot.finalize_rpgeo_extension(ROOT)
    print(json.dumps(decision,indent=2))
    display(pd.read_csv(ROOT/'rpgeo_method_summary.csv'))
else:
    decision = result
    print('Exact-face RP-Geo training correctly stopped:', gate['decision'])
    probe_path = ROOT/'rpgeo_retention_probe/rpgeo_retention_pareto.csv'
    if probe_path.is_file():
        display(pd.read_csv(probe_path))
        print(json.loads((ROOT/'rpgeo_retention_probe/rpgeo_retention_probe.json').read_text()))

## Export the complete resumable extension

In [ ]:
required = [
 'rpgeo_extension_protocol.json', 'rpgeo_offline_gate/rpgeo_offline_gate.json',
 'rpgeo_offline_gate/rpgeo_offline_gate.csv',
 'uniform/checkpoints/epoch_100.pt', 'uniform/training_provenance.json',
]
if gate['decision'] == 'GO':
    required += [
      'resource_geo/checkpoints/epoch_100.pt','resource_geo/training_provenance.json',
      'resource_geo/rpgeo_refresh_metrics.csv','rpgeo_method_summary.csv',
      'rpgeo_trajectory_variance_diagnostics.csv','rpgeo_summary.json',
    ]
else:
    required += [
      'rpgeo_retention_probe/rpgeo_retention_probe.json',
      'rpgeo_retention_probe/rpgeo_retention_pareto.csv',
      'rpgeo_retention_probe/rpgeo_retention_probe_by_state.csv',
    ]
missing = [name for name in required if not (ROOT/name).is_file() or (ROOT/name).stat().st_size == 0]
assert not missing, f'Missing extension artifacts: {missing}'
bundle = Path('/kaggle/working/rq2-rpgeo-extension-v1.zip')
with zipfile.ZipFile(bundle,'w',compression=zipfile.ZIP_DEFLATED,allowZip64=True) as archive:
    for path in ROOT.rglob('*'):
        if path.is_file(): archive.write(path,Path('e2e_pairwise_pilot_v2')/path.relative_to(ROOT))
print('Download/persist:',bundle,f'{bundle.stat().st_size/2**30:.2f} GiB')
bundle